In [2]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

df = pd.read_csv("../data/titanic.csv")

df.shape, df.head()


((891, 12),
    PassengerId  Survived  Pclass  \
 0            1         0       3   
 1            2         1       1   
 2            3         1       3   
 3            4         1       1   
 4            5         0       3   
 
                                                 Name     Sex   Age  SibSp  \
 0                            Braund, Mr. Owen Harris    male  22.0      1   
 1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
 2                             Heikkinen, Miss. Laina  female  26.0      0   
 3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
 4                           Allen, Mr. William Henry    male  35.0      0   
 
    Parch            Ticket     Fare Cabin Embarked  
 0      0         A/5 21171   7.2500   NaN        S  
 1      0          PC 17599  71.2833   C85        C  
 2      0  STON/O2. 3101282   7.9250   NaN        S  
 3      0            113803  53.1000  C123        S  
 4      0            373

In [3]:
id_cols = ["PassengerId", "Name", "Ticket"]
target_col = "Survived"

feature_cols = [col for col in df.columns if col not in id_cols + [target_col]]


In [4]:
numerical_cols = df[feature_cols].select_dtypes(include=np.number).columns.tolist()
categorical_cols = df[feature_cols].select_dtypes(exclude=np.number).columns.tolist()

numerical_cols, categorical_cols


(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], ['Sex', 'Cabin', 'Embarked'])

In [5]:
missing_pct = (df[feature_cols].isnull().mean() * 100).round(2).to_dict()

high_missing_cols = [col for col, pct in missing_pct.items() if pct > 30]
moderate_missing_cols = [col for col, pct in missing_pct.items() if 5 < pct <= 30]


In [6]:
cardinality = {col: df[col].nunique() for col in categorical_cols}

high_cardinality_cols = [col for col, val in cardinality.items() if val > 10]
low_cardinality_cols = [col for col, val in cardinality.items() if val <= 10]


In [7]:
skewness = {}
kurtosis_vals = {}

for col in numerical_cols:
    skewness[col] = round(skew(df[col].dropna()), 3)
    kurtosis_vals[col] = round(kurtosis(df[col].dropna()), 3)

high_skew_cols = [col for col, val in skewness.items() if abs(val) > 1]


In [8]:
corr_matrix = df[numerical_cols].corr()

high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.75:
            high_corr_pairs.append(
                (corr_matrix.columns[i], corr_matrix.columns[j],
                 round(corr_matrix.iloc[i, j], 2))
            )


In [9]:
target_distribution = df[target_col].value_counts(normalize=True).round(2).to_dict()

task_type = "classification" if df[target_col].nunique() <= 10 else "regression"


In [10]:
eda_summary = {
    "dataset_shape": df.shape,
    "target_column": target_col,
    "task_type": task_type,

    "numerical_columns": numerical_cols,
    "categorical_columns": categorical_cols,

    "missing_percentage": missing_pct,
    "high_missing_columns": high_missing_cols,
    "moderate_missing_columns": moderate_missing_cols,

    "categorical_cardinality": cardinality,
    "high_cardinality_columns": high_cardinality_cols,

    "skewness": skewness,
    "high_skew_columns": high_skew_cols,

    "correlated_pairs": high_corr_pairs,

    "target_distribution": target_distribution
}


In [11]:
eda_summary


{'dataset_shape': (891, 12),
 'target_column': 'Survived',
 'task_type': 'classification',
 'numerical_columns': ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'],
 'categorical_columns': ['Sex', 'Cabin', 'Embarked'],
 'missing_percentage': {'Pclass': 0.0,
  'Sex': 0.0,
  'Age': 19.87,
  'SibSp': 0.0,
  'Parch': 0.0,
  'Fare': 0.0,
  'Cabin': 77.1,
  'Embarked': 0.22},
 'high_missing_columns': ['Cabin'],
 'moderate_missing_columns': ['Age'],
 'categorical_cardinality': {'Sex': 2, 'Cabin': 147, 'Embarked': 3},
 'high_cardinality_columns': ['Cabin'],
 'skewness': {'Pclass': np.float64(-0.629),
  'Age': np.float64(0.388),
  'SibSp': np.float64(3.689),
  'Parch': np.float64(2.744),
  'Fare': np.float64(4.779)},
 'high_skew_columns': ['SibSp', 'Parch', 'Fare'],
 'correlated_pairs': [],
 'target_distribution': {0: 0.62, 1: 0.38}}

In [12]:
import json

with open("../outputs/eda_summary.json", "w") as f:
    json.dump(eda_summary, f, indent=4)
